## 0) Why Regex Matters in NLP (Motivation)

Regex (Regular Expressions) is not an alternative to Machine Learning.<br> It is a **precision tool** used before, beside, and sometimes instead of ML.

In real NLP pipelines, regex is commonly used for:

- Cleaning noisy text (URLs, HTML, emojis, extra spaces)
- Extracting structured patterns (emails, phone numbers, dates)
- Rule-based signals added as ML features
- Fast heuristics where patterns are well-defined

*Example:*

- **Scenario**: You have a 10,000-page document and need to find every email address. You can't search for "gmail" because there are "outlook," "yahoo," and custom domains.
- **The Solution**: You need a pattern, not a word.

## 1) Regex in Python: The `re` Module

We use Python’s built-in `re` module.

Core functions we will use in NLP:

- `re.search()` – 	Returns a Match object if there is a match anywhere in the string
- `re.findall()` – extract all matches
- `re.sub()` – replace patterns (cleaning)
- `re.split()` – rule-based splitting
- `re.match()` – find a pattern at the beginning of a string
- `re.finditer()` – find all matches as an iterator


In [ ]:
import re
print("re.search: ", re.search(pattern=r'\bfoo\b', string='bar baz\nfoo', flags=re.MULTILINE))  # Matches 'foo' as a whole word
print("re.findall: ", re.findall(r'\Ba\B', 'foo bar baz')) 
print("re.sub: ", re.sub(r'\bfoo\b', 'bar', 'foo bar baz'))
print("re.split: ", re.split(r'\s', 'foo bar baz'))
print("re.match: ", re.match(r'\bfoo\b', 'bar baz\nfoo')) 
print("re.finditer: ", list(re.finditer(r'\bfoo\b', 'foo bar baz foo')))

re.search:  <re.Match object; span=(8, 11), match='foo'>
re.findall:  ['a', 'a']
re.sub:  bar bar baz
re.split:  ['foo', 'bar', 'baz']
re.match:  None
re.finditer:  [<re.Match object; span=(0, 3), match='foo'>, <re.Match object; span=(12, 15), match='foo'>]


In python, `iterator` is a object that can be used to iterate over a sequence of values, such as a list or a string. 
- They do not store all the values in memory at once, but instead generate them on-the-fly as you iterate over them. 
- This makes iterators more memory-efficient than lists, especially when dealing with large datasets.

In [195]:
print(re.finditer(r'\bfoo\b', 'foo bar baz foo'))

## 2) Anchors

Anchors match a position before or after other characters.

| Syntax | Description | Example pattern | Example matches | Example non-matches |
| :--- | :--- | :--- | :--- | :--- |
| `^` | match start of line | `^r` | <span style="background-color: #6de0d1;">r</span>abbit<br><span style="background-color: #6de0d1;">r</span>accoon | parrot<br>ferret |
| `$` | match end of line | `t$` | rabbi<span style="background-color: #6de0d1;">t</span><br>foo<span style="background-color: #6de0d1;">t</span> | trap<br>star |
| `\A` | match start of line | `\Ar` | <span style="background-color: #6de0d1;">r</span>abbit<br><span style="background-color: #6de0d1;">r</span>accoon | parrot<br>ferret |
| `\Z` | match end of line | `t\Z` | rabbi<span style="background-color: #6de0d1;">t</span><br>foo<span style="background-color: #6de0d1;">t</span> | trap<br>star |
| `\b` | match characters at the start or end of a word | `\bfox\b` | the red <span style="background-color: #6de0d1;">fox</span> ran<br>the <span style="background-color: #6de0d1;">fox</span> ate | foxtrot<br>foxskin scarf |
| `\B` | match characters in the middle of other non-space characters | `\Bee\B` | tr<span style="background-color: #6de0d1;">ee</span>s<br>r<span style="background-color: #6de0d1;">ee</span>f | bee<br>tree |

### 2.1) `^` and `\A`

Difference: `^` can match after newline in multiline mode (`re.MULTILINE`), while `\A` always matches only the absolute start.

In [82]:
import re

text1 = "first line\nsecond line"
print(repr(text1))

'first line\nsecond line'


In [57]:
pattern1 = r'^second'  # With multiline flag
pattern2 = r'\Asecond'  # Never matches after newline

# With re.MULTILINE
print(re.search(pattern=pattern1, string=text1, flags=re.MULTILINE))  # Matches "second"
print(re.search(pattern=pattern2, string=text1, flags=re.MULTILINE))  # No match

<re.Match object; span=(11, 17), match='second'>
None


In [58]:
# Without flags
print(re.search(pattern=pattern1, string=text1))  # No match
print(re.search(pattern=pattern2, string=text1))  # No match

None
None


### 2.2) `$` vs `\Z` (End of line/string)
Difference: `$` can match before newline in multiline mode, `\Z` matches absolute end only.

In [83]:
text2 = "first line\nsecond line\n"
pattern1 = r'line$'  # With multiline flag
pattern2 = r'line\Z'  # Only at absolute end
print(repr(text2))

'first line\nsecond line\n'


In [61]:
# With re.MULTILINE
print(re.search(pattern=pattern1, string=text2, flags=re.MULTILINE))  # Matches first "line"
print(re.search(pattern=pattern2, string=text2, flags=re.MULTILINE))  # No match (trailing newline)

<re.Match object; span=(6, 10), match='line'>
None


In [62]:
# Match exact end
pattern3 = r'line\n\Z'
print(re.search(pattern=pattern3, string=text2, flags=re.MULTILINE))  # Matches!

<re.Match object; span=(18, 23), match='line\n'>


### 2.3) `\b` (Word boundary)

Difference: Matches transition between word (`\w`) and non-word (`\W`) characters.

In [ ]:
text3 = "fox foxes foxtrot"
pattern = r'\bfox\b'
display(text3)

matches: list[any] = re.findall(pattern=pattern, string=text3)
print(matches)  # ['fox'] only
# 'foxes' fails because 's' is word character
# 'foxtrot' fails because 'x' is part of longer word

'fox foxes foxtrot'

['fox']


### 2.4) `\B` (Non-word boundary)
Difference: Opposite of \b - matches where \b wouldn't.

In [ ]:
text4 = "trees freeze bee see"
pattern = r'\Bee\B'
display(text4)

matches: list[any] = re.findall(pattern=pattern, string=text4)
print(matches)  # ['ee', 'ee'] from 'trees' and 'freeze'
# 'bee' doesn't match - 'ee' at end
# 'see' doesn't match - 'ee' at end

'trees freeze bee see'

['ee', 'ee']


## 3) Matching types of character

Rather than matching specific characters, we can match specific types of characters such as letters, numbers, and more.

| Syntax | Description | Example pattern | Example matches | Example non-matches |
| :--- | :--- | :--- | :--- | :--- |
| `.` | anything except for a linebreak, <br> matches 0 or more item | `c.e` | <span style="background-color: #6de0d1;">cle</span>an<br><span style="background-color: #6de0d1;">che</span>ap | acert<br>cent |
| `\d` | match a digit | `\d` | <span style="background-color: #6de0d1;">6060</span>-<span style="background-color: #6de0d1;">842</span><br><span style="background-color: #6de0d1;">2</span>b\|^<span style="background-color: #6de0d1;">2</span>b | two<br>**___ |
| `\D` | Match a non-digit | `\D` | <span style="background-color: #6de0d1;">The</span> 5 <span style="background-color: #6de0d1;">cats ate</span><br>12 <span style="background-color: #6de0d1;">Angry men</span> | 52<br>10032 |
| `\w` | Match word characters | `\wee\w` | t<span style="background-color: #6de0d1;">rees</span><br><span style="background-color: #6de0d1;">bee4</span> | The bee<br>eels eat meat |
| `\W` | Match non-word characters | `\Wbat\W` | At <span style="background-color: #6de0d1;">bat</span><br>Swing the <span style="background-color: #6de0d1;">bat</span> fast | wombat<br>bat53 |
| `\s` | Match whitespace | `\sfox\s` | the <span style="background-color: #6de0d1;">fox</span> ate<br>his <span style="background-color: #6de0d1;">fox</span> ran | it’s the fox.<br>foxfur |
| `\S` | Match non-whitespace | `\See\S` | t<span style="background-color: #6de0d1;">rees</span><br><span style="background-color: #6de0d1;">reef</span> | the bee stung<br>The tall tree |
| `\metacharacter` | Escape a metacharacter | `\.`<br>`\^`<br> `\+` | The cat ate<span style="background-color: #6de0d1;">.</span><br>2<span style="background-color: #6de0d1;">^</span>3<br>C<span style="background-color: #6de0d1;">++</span> | the cat ate<br>23<br>C |


### 3.1) `.` (Dot) 
Match any character except newline

In [100]:
import re
text = "cat\ncup cop\nc@t\nct c"
pattern = r'c.t'  # c, any char, t
print("text:", repr(text))

matches: list[any] = re.findall(pattern=pattern, string=text)
print(f"'c.t' matches: {matches}") 
# Note: 'c\nt' would not match because . doesn't match newline

text: 'cat\ncup cop\nc@t\nct c'
'c.t' matches: ['cat', 'c@t']


### 3.2) `\d`
Match a digit

In [104]:
text = "Phone: 555-1234, Age: 2, Price: $19.99"
pattern = r'\d'

matches: list[any] = re.findall(pattern=pattern, string=text)
print(f"\\d matches: {matches}")  # ['5', '5', '5', '1', '2', '3', '4', '2', '5', '1', '9', '9', '9']

\d matches: ['5', '5', '5', '1', '2', '3', '4', '2', '1', '9', '9', '9']


### 3.3) `\D` 
Match non-digit

In [102]:
text = "Room 101 opens at 9:00 AM"
pattern = r'\D'
print(repr(text))
matches = re.findall(pattern, text)
print(f"\\D matches: {matches}")  

'Room 101 opens at 9:00 AM'
\D matches: ['R', 'o', 'o', 'm', ' ', ' ', 'o', 'p', 'e', 'n', 's', ' ', 'a', 't', ' ', ':', ' ', 'A', 'M']


### 3.4) `\w` 
Match word character (letters, digits, underscore)

In [103]:
text = "User_123: active passives, Price: $99.99"
pattern = r'\w'
print(repr(text))
matches = re.findall(pattern, text)
print(f"\\w matches: {matches}")  

matches = re.findall(r'\wve', text)
print(f"\\wve matches: {matches}")  

'User_123: active passives, Price: $99.99'
\w matches: ['U', 's', 'e', 'r', '_', '1', '2', '3', 'a', 'c', 't', 'i', 'v', 'e', 'p', 'a', 's', 's', 'i', 'v', 'e', 's', 'P', 'r', 'i', 'c', 'e', '9', '9', '9', '9']
\wve matches: ['ive', 'ive']


### 3.5) `\W` 
Match non-word character

In [101]:
text = "User_123... active Price: $99.99"
pattern = r'\W'
print("Text:", repr(text))

matches = re.findall(pattern, text)
print(f"\\W matches: {matches}")  

Text: 'User_123... active Price: $99.99'
\W matches: ['.', '.', '.', ' ', ' ', ':', ' ', '$', '.']


### 3.6) `\s` 
Match whitespace characters.

In [105]:
text = "Hello\tworld\nPython 3.9"
pattern = r'\s'
print("Text:", repr(text))

matches = re.findall(pattern, text)
print(f"\\s matches: {matches}")  # ['\t', '\n', ' ']

Text: 'Hello\tworld\nPython 3.9'
\s matches: ['\t', '\n', ' ']


### 3.7) `\S` 
Match non-whitespace characters.

In [106]:
text = "Hello world\nPython\t3.9 $"
pattern = r'\S'
print("Text:", repr(text))

matches = re.findall(pattern, text)
print(f"\\S matches: {matches}")  


Text: 'Hello world\nPython\t3.9 $'
\S matches: ['H', 'e', 'l', 'l', 'o', 'w', 'o', 'r', 'l', 'd', 'P', 'y', 't', 'h', 'o', 'n', '3', '.', '9', '$']


### 3.8) `\` (Backslash) 
Escape metacharacters

In [62]:
text = r"Special chars: . * + ? ^ $ ( ) [ ] { } | \\"
print("Text:", repr(text))
pattern = r'\.'  # Match literal dot

matches = re.findall(pattern, text)
print(f"\\. matches: {matches}") 
print('--'*25)

text2 = r"Price: $19.99, Email: user@example.com, ^Start \End"
print("Text2:", repr(text2))
pattern2 = r'\$|\.|\@|\^|\\'  
matches2 = re.findall(pattern2, text2)
print(f"\\$ \\. \\@ \\^ \\ matches: {matches2}")  


Text: 'Special chars: . * + ? ^ $ ( ) [ ] { } | \\\\'
\. matches: ['.']
--------------------------------------------------
Text2: 'Price: $19.99, Email: user@example.com, ^Start \\End'
\$ \. \@ \^ \ matches: ['$', '.', '@', '.', '^', '\\']


## 4) Quantifiers Table

Rather than matching single instances of characters, we can match repeated characters.

| Syntax | Description | Example pattern | Example matches | Example non-matches |
| :--- | :--- | :--- | :--- | :--- |
| `x*` | match zero or more times | `ar*o` | cac<span style="background-color: #6de0d1;">ao</span><br>c<span style="background-color: #6de0d1;">arro</span>t | arugula<br>artichoke |
| `x+` | match one or more times | `re+` | g<span style="background-color: #6de0d1;">ree</span>n<br>t<span style="background-color: #6de0d1;">ree</span> | trap<br>ruined |
| `x?` | Match zero or one times | `ro?a` | <span style="background-color: #6de0d1;">roa</span>st<br><span style="background-color: #6de0d1;">ra</span>nt | root<br>rear |
| `x{m}` | match m times | `\we{2}\w` | <span style="background-color: #6de0d1;">deer</span><br><span style="background-color: #6de0d1;">seer</span> | red<br>enter |
| `x{m,}` | match m or more times | `2{3,}4` | 671-<span style="background-color: #6de0d1;">2224</span><br><span style="background-color: #6de0d1;">2222224</span> | 224<br>123 |
| `x{m,n}` | match between m and n times | `12{1,3}3` | <span style="background-color: #6de0d1;">123</span>4<br><span style="background-color: #6de0d1;">12223</span>84 | 15335<br>1222223 |
| `x*?, x+?, etc.` | match the minimum number of times - known as a lazy quantifier | `re+?` | t<span style="background-color: #6de0d1;">re</span>e<br>f<span style="background-color: #6de0d1;">re</span>eeee | trout<br>roasted |



### 4.1) `*`
matches zero or more times

In [114]:
text1 = "cao carrot arugula"
matches = re.findall(r'ar*o', text1)
print(f"Pattern: 'ar*o'")
print(f"Matches: {matches}") 

Pattern: 'ar*o'
Matches: ['ao', 'arro']


### 4.2) `+`
matches one or more times

In [162]:
text2 = "green treee gre gr" 
matches = re.findall(r're+', text2)
print(f"Pattern: 're+'")
print(f"Matches: {matches}")

Pattern: 're+'
Matches: ['ree', 'reee', 're']


### 4.3) `?`
matches zero or one times

In [116]:
text3 = "roast rant root"
matches = re.findall(r'ro?a', text3)
print(f"Pattern: 'ro?a'")
print(f"Matches: {matches}")  

Pattern: 'ro?a'
Matches: ['roa', 'ra']


### 4.4) `{m}`
matches exactly m times

In [119]:
text4 = "deer seer red eenter"
matches = re.findall(r'\we{2}\w', text4)
print(f"Pattern: '\\we{{2}}\\w'")
print(f"Matches: {matches}") 

Pattern: '\we{2}\w'
Matches: ['deer', 'seer']


### 4.5) `{m,}`
matches m or more times

In [123]:
text5 = "6712224 2222224 224 123"
matches = re.findall(r'2{3,}4', text5)
print(f"Pattern: '2{{3,}}4'")
print(f"Matches: {matches}")

Pattern: '2{3,}4'
Matches: ['2224', '2222224']


### 4.6) `{m,n}`
matches between m and n times

In [125]:
text6 = "1234 1222384 1222223"
matches = re.findall(r'12{1,3}3', text6)
print(f"Pattern: '12{{1,3}}3'")
print(f"Matches: {matches}")

Pattern: '12{1,3}3'
Matches: ['123', '12223']


### 4.7) `*?`, `+?`, `??`
matches the minimum number of times - known as a lazy quantifier

In [140]:
text7 = "tree freeeeee fre trout roasted"
matches = re.findall(r're+?', text7)
print(f"Pattern: 're+?' (lazy)")
print(f"Matches: {matches}")

Pattern: 're+?' (lazy)
Matches: ['re', 're', 're']


### 4.8) Greedy vs. Lazy Quantifiers
matches as much as possible (greedy) vs. as little as possible (lazy)

In [132]:
text8 = "<div>content</div><div>more</div><body>againcontent</body>"
print("--- Greedy vs Lazy comparison---")
greedy = re.findall(r'<div>.*</div>', text8)
lazy = re.findall(r'<div>.*?</div>', text8)
print(f"Text: '{text8}'")
print(f"Greedy '<div>.*</div>': {greedy}")  
print(f"Lazy '<div>.*?</div>': {lazy}")    

--- Greedy vs Lazy comparison---
Text: '<div>content</div><div>more</div><body>againcontent</body>'
Greedy '<div>.*</div>': ['<div>content</div><div>more</div>']
Lazy '<div>.*?</div>': ['<div>content</div>', '<div>more</div>']


## 5) Questions for practice:

**Q1:** Write a regex pattern to find all sequences of digits in the string: `"Room 404 opens at 9:30 AM"`<br>
*Output*: `['404', '9', '30']

**Q2:** Create a pattern to match words that start with `c` and end with `t` in: `"cat cot cut coat carts"`<br>
*Output*: `['cat', 'cot', 'cut', 'coat']`

**Q3:** Extract all hashtags from: `"Learning #Python #regex #fun #100DaysOfCode"`<br>
*Output*: `['#Python', '#regex', '#fun', '#100DaysOfCode']`

**Q4:** Create a pattern to find all standalone 'fox' words (not part of other words) in: `"fox foxfire foxtrot red fox"`<br>
*Output*: `['fox', 'fox']`

**Q5:** Find email domains in: "Emails: user@example.com, admin@site.org, test@company.co.uk"<br>
*Output*: `['@example.com', '@site.org', '@company.co.uk']`

## 6) Character Classes Table

- **Square Brackets `[ ]` = Character Class**<br>
    Means "match ANY ONE of the characters inside"

- **With Caret ^ Inside `[ ]` = Negation**<br>
    `[^abc]` means: match ANY character that is NOT a, b, or c

    The caret `^` inside brackets means "NOT" or "everything except"


| Python Syntax | Description | Example Pattern | Example Matches | Example Non-Matches |
|--------------|-------------|-----------------|-----------------|---------------------|
| `[abc]` | **Character class** - match any one of the characters inside | `r'[aeiou]'` | <span style="background-color: #6de0d1;">a</span>ppl<span style="background-color: #6de0d1;">e</span><br>h<span style="background-color: #6de0d1;">e</span>ll<span style="background-color: #6de0d1;">o</span> | rhythm<br>xyz |
| `[a-z]` | **Character range** - match any lowercase letter | `r'[a-z]{3}'` | <span style="background-color: #6de0d1;">cat</span><br><span style="background-color: #6de0d1;">dog</span> | CAT<br>123 |
| `[A-Z]` | **Uppercase range** - match any uppercase letter | `r'[A-Z]+'` | <span style="background-color: #6de0d1;">HELLO</span> world | hello<br>123 |
| `[0-9]` | **Digit range** - match any digit | `r'[0-9]{3}'` | <span style="background-color: #6de0d1;">123</span>-45 | abc<br>12 |
| `[a-zA-Z]` | **Combined ranges** - match any letter | `r'[a-zA-Z]+'` | <span style="background-color: #6de0d1;">Hello</span> <span style="background-color: #6de0d1;">World</span> | 123<br>hello123 |
| `[^abc]` | **Negated character class** - match any character EXCEPT these | `r'[^aeiou\s]+'` | <span style="background-color: #6de0d1;">ct</span><br><span style="background-color: #6de0d1;">dg</span> | a<br>e<br>(space) |
| `[^a-z]` | **Negated range** - match anything except lowercase letters | `r'[^a-z]+'` | <span style="background-color: #6de0d1;">ABC123</span> | hello<br>world |
| `[^0-9]` | **Non-digit characters** | `r'[^0-9]+'` | <span style="background-color: #6de0d1;">Phone: </span> | 123<br>456 |
| `[\w\s]` | **Union of classes** - match word chars OR whitespace | `r'[\w\s]+'` | <span style="background-color: #6de0d1;">Hello World 123</span> | @#$%<br>.!, |
| `[a-f0-9]` | **Hexadecimal characters** | `r'[a-f0-9]+'` | <span style="background-color: #6de0d1;">abc123</span><br><span style="background-color: #6de0d1;">deadbeeze</span> | xyz<br>ghijk |
| `[.\-]` | **Escaped special characters** - match literal dot or dash | `r'[\w.\-]+'` | <span style="background-color: #6de0d1;">user.name-01</span> | user@name<br>name space |

### 6.1) `[abc]` - Character class

In [47]:
text1 = "apple orange banana"
matches = re.findall(r'[aeiou]', text1)
print(f"'[aeiou]' in '{text1}': {matches}")

'[aeiou]' in 'apple orange banana': ['a', 'e', 'o', 'a', 'e', 'a', 'a', 'a']


### 6.2) `[a-z]` - Lowercase range

In [46]:
text2 = "Hello World 123"
matches = re.findall(r'[a-z]+', text2)
print(f"'[a-z]+' in '{text2}': {matches}")

'[a-z]+' in 'Hello World 123': ['ello', 'orld']


### 6.3) `[A-Z]` - Uppercase range

In [48]:
text3 = "Hello World 123"
matches = re.findall(r'[A-Z]+', text3)
print(f"'[A-Z]+' in '{text3}': {matches}")

'[A-Z]+' in 'Hello World 123': ['H', 'W']


### 6.4) `[0-9]` - Digit range

In [49]:
text4 = "Phone: 555-1234"
matches = re.findall(r'[0-9]+', text4)
print(f"   '[0-9]+' in '{text4}': {matches}")

   '[0-9]+' in 'Phone: 555-1234': ['555', '1234']


### 6.5) `[a-zA-Z]` - Uppercase range

In [51]:
text3 = "Hello World 123"
matches = re.findall(r'[a-zA-Z]+', text3)
print(f"'[a-zA-Z]+' in '{text3}': {matches}")

'[a-zA-Z]+' in 'Hello World 123': ['Hello', 'World']


### 6.6) `[^abc]` - Negated character class

In [57]:
text6 = "user.name-01@example.com"
matches = re.findall(r'[^aeiou\s]', text6)
print(f"'[^aeiou\\s]' in '{text6}': {matches}")

'[^aeiou\s]' in 'user.name-01@example.com': ['s', 'r', '.', 'n', 'm', '-', '0', '1', '@', 'x', 'm', 'p', 'l', '.', 'c', 'm']


In [78]:
# Check for properly closed tags
html = "<h1>Title</h1>"
pattern = r'[^<]+'

matches = list(re.finditer(pattern, html))
print(f"Properly closed tags: {matches}")
# Output: ['h1'] - only <h1> is properly closed

Properly closed tags: [<re.Match object; span=(1, 9), match='h1>Title'>, <re.Match object; span=(10, 14), match='/h1>'>]


In [ ]:
html = "<h1>Title</h1>"
match = list(re.finditer(r'>([^></])+<', html))
match


[<re.Match object; span=(3, 10), match='>Title<'>]

### 6.7) `[^a-z]` - NOT lowercase

In [59]:
text3 = "cat DOG Bird 123"
matches = re.findall(r'[^a-z]+', text3)
print(f"'[^a-z]+' in '{text3}': {matches}")

'[^a-z]+' in 'cat DOG Bird 123': [' DOG B', ' 123']


### 6.8) `[^0-9]` - Non-digits

In [61]:
text4 = "Phone: 555-1234"
matches = re.findall(r'[^0-9]+', text4)
print(f"'[^0-9]+' in '{text4}': {matches}")

'[^0-9]+' in 'Phone: 555-1234': ['Phone: ', '-']


### 6.9) `[\w\s]` - Word chars or whitespace

In [64]:
text2 = "Hello World @123"
matches = re.findall(r'[\w\s]+', text2)
print(f"'[\\w\\s]+' in '{text2}': {matches}")

'[\w\s]+' in 'Hello World @123': ['Hello World ', '123']


### 6.10) `[a-f0-9]` - Hexadecimal chars

In [66]:
text5 = "abcg123 XYZ789"
matches = re.findall(r'[a-f0-9]+', text5, re.IGNORECASE)
print(f"   '[a-f0-9]+' (case-insensitive) in '{text5}': {matches}")

   '[a-f0-9]+' (case-insensitive) in 'abcg123 XYZ789': ['abc', '123', '789']


### 6.11) `[.\-]` - Escaped special chars

In [71]:
text6 = "user.name-01@example.com"
matches = re.findall(r'[\w\.\-]+', text6)
print(f"'[\\w.\\-]+' in '{text6}': {matches}")

'[\w.\-]+' in 'user.name-01@example.com': ['user.name-01', 'example.com']


## 7) Groups Table
Grouping in Python regular expressions allows you to treat a part of a pattern as a single unit and to extract specific sections of the matched text. Groups are defined by enclosing the sub-pattern in parentheses `()`. 

| Python Syntax      | Description                                         | Example Pattern                     | Example Matches                   | Example Non-Matches |
| ------------------ | --------------------------------------------------- | ----------------------------------- | --------------------------------- | ------------------- |
| `(pattern)`              | **Capturing group** – captures the matched text     | `r'(iss)'`                          | M<span style="background-color: #6de0d1;">ississ</span>ippi<br>m<span style="background-color: #6de0d1;">iss</span>ed     | mist<br>persist     |
| `(pattern)+`       | **Grouped repetition**                              | `r'(ab)+'`                          | <span style="background-color: #6de0d1;">ab</span>, <span style="background-color: #6de0d1;">abab</span>                  | a<br>b              |
| `(?:pattern)`            | **Non-capturing group** – groups without capturing  | `r'(?:ab)(cd)'`                     | <span style="background-color: #6de0d1;">abcd</span><br>Group 1: <span style="background-color: #6de0d1;">cd</span>       | acbd                |
| `(?P<name>x)`      | **Named capturing group**                           | `r'(?P<first>\d)(?P<second>\d)\d*'` | <span style="background-color: #6de0d1;">1325</span><br>first: 1<br>second: 3 | 2<br>hello          |
| `(x\|y)`           | **Alternation** – match one of multiple patterns    | `r'(re\|ba)'`                       | <span style="background-color: #6de0d1;">re</span>d<br><span style="background-color: #6de0d1;">ba</span>nther             | rant<br>bear        |
| `\1`, `\2`, …      | **Numeric backreference** – refer to earlier groups | `r'(b)(\w*)\1'`                     | <span style="background-color: #6de0d1;">blob</span><br><span style="background-color: #6de0d1;">brib</span>e             | bear<br>bring       |
| `(?P=name)`        | **Named backreference** – refer to named group      | `r'(?P<first>5)\d*(?P=first)'`      | <span style="background-color: #6de0d1;">51245</span><br><span style="background-color: #6de0d1;">55</span>               | 523<br>51           |
| `(?:pattern)+`     | **Repeated non-capturing group**                    | `r'(?:ab)+'`                        | <span style="background-color: #6de0d1;">ab</span>, <span style="background-color: #6de0d1;">abab</span>                  | aba                 |
| `(\w)\1`           | **Immediate repetition** using backreference        | `r'(\w)\1'`                         | <span style="background-color: #6de0d1;">ll</span> in hello<br><span style="background-color: #6de0d1;">ss</span> in miss | abc                 |


#### **Question 1: Why grouping needed in regex?**
**Answer:** Grouping exists so regex can remember, reuse, and structure information.
Without groups, `re` is just a fancy `find()`.

> *Example: 1*

Detect duplicated words (common NLP cleanup)

```python
text = "this is is a test test sentence"
pattern = r"\b(\w+)\s+\1\b"

matches = re.findall(pattern, text)
print(matches)
```

Output:
```
['is', 'test']
```

> *Example: 2*

Parse log files

```python
text = "ERROR 2025-08-21 Disk full"
pattern = r"(?P<level>ERROR|INFO|DEBUG)\s+(?P<date>\d{4}-\d{2}-\d{2})\s+(?P<msg>.+)"

match = re.search(pattern, text)

print(match.groupdict())
```

Output:
```
{'level': 'ERROR', 'date': '2025-08-21', 'msg': 'Disk full'}
```


#### **Question 2: What is a capturing group in regex and why is it useful?**<br>
**Answer:** A `capturing group` in regular expressions is a way to treat multiple characters as a single unit by enclosing them in parentheses.<br>
This serves two primary functions: 

- *Grouping*: It allows you to apply quantifiers (like `+`, `*` , or `{n}`) to an entire subpattern rather than just the preceding character. For example,  `(ha)+` matches "haha", "hahaha", and "hahahaha", whereas `ha+` matches "haa", "haaa", etc.

- *Capturing (Memorizing)*: It saves the portion of the input string that matches the subpattern into memory e.g., `match.group(1)`, `match.group(2)`. <br>The engine stores: 
- - the start index
- - the end index
- - the matched text

This captured text can then be accessed later for various purposes, such as: 

- - - *Backreferencing* within the same regular expression using `\1`, `\2`, etc., to match the exact same text again. For example, `(\w+)\s+\1` finds duplicated words like "the the". 

- - - *Extracting specific data* from a string programmatically. For instance, we could extract the day, month, and year from a date string like "23-05-2023".

- - - *Replacement* in search-and-replace operations using `$1`, `$2`, or named references like `$(name)` in the replacement string. 

#### **Question 3: What is different types of groups in regex?**<br>
**Answer:** There are three types of groups in regular expressions:

- **Numbered Capturing Groups**: These are the standard groups created with `(....)` and are numbered automatically from left to right based on their opening parenthesis, starting from `1`. `Group 0` always refers to the entire match. They possess group functionality e.g., `matches.group(0)` or `matches.group(1)`.

- **Named Capturing Groups**: Supported in many modern regex flavors (e.g., Python, .NET, JavaScript with 'd' flag), these groups use syntax like `(?P<name>...)` . They offer improved readability and maintenance by allowing access to the captured data by name. 

- **Non-Capturing Groups**: Created using `(?:....)`, these groups are used purely for grouping a subpattern to apply a quantifier or alternation, but they do not store the matched text in memory and do not count toward group numbering. This can offer minor performance benefits and prevents refactoring issues if the order of groups changes. They do not possess group functionality e.g., `matches.group(0)` or `matches.group(1)`.



### 7.1) `(pattern)` and `(pattern)+`
**Named capturing group**: matches with capturing groups and grouped repetition

In [244]:
text1 = "Missed mississipi mischief"
pattern1 = r'(iss)'
pattern2 = r'(iss)+'
matches1 = re.findall(pattern1, text1)
matches2 = re.findall(pattern2, text1)
full_matches = list(re.finditer(pattern2, text1))
print(f"{pattern1} matches: {matches1}")
print(f"{pattern2} matches: {matches2}")
print(f"{pattern2} Full matches: {full_matches}")

(iss) matches: ['iss', 'iss', 'iss']
(iss)+ matches: ['iss', 'iss']
(iss)+ Full matches: [<re.Match object; span=(1, 4), match='iss'>, <re.Match object; span=(8, 14), match='ississ'>]


**Que. Why Does `findall()` Only Return "iss" for "ississ"?**<p>
**Ans.** Because with repeating capturing groups `(iss)+`:

- Each time `(iss)` matches, it overwrites the previous capture.

- When matching `"ississ"`:

- - First `"iss"` → group = `"iss"`

- - Second `"iss"` → group = `"iss"` (overwrites)

- - Final captured value: `"iss"` (the last one)

Regex does not keep a history. If it stored everything, regex would turn into a parser.

So in summary:

- `(iss)` finds 3 separate `"iss"` patterns.

- `(iss)+` finds 2 matches: one `"iss"` and one `"ississ"`, but `findall()` only returns the captured group (`'iss'` for both).

In [222]:
text = "abababab ab abab"
pattern1 = r'(ab)'
pattern2 = r'(ab)+'

print("Text:", text)
print()

print("1. (ab) - Simple capturing:")
matches1 = re.findall(pattern1, text)
print(f"   Result: {matches1}") 

print("\n2. (ab)+ - Repeating capturing group:")
matches2 = re.findall(pattern2, text)
print(f"   Result: {matches2}") 

print("\n3. What's really matched with (ab)+:")
for match in re.finditer(pattern2, text):
    print(f"   Full: '{match.group(0)}', Captured: '{match.group(1)}'")

Text: abababab ab abab

1. (ab) - Simple capturing:
   Result: ['ab', 'ab', 'ab', 'ab', 'ab', 'ab', 'ab']

2. (ab)+ - Repeating capturing group:
   Result: ['ab', 'ab', 'ab']

3. What's really matched with (ab)+:
   Full: 'abababab', Captured: 'ab'
   Full: 'ab', Captured: 'ab'
   Full: 'abab', Captured: 'ab'


- The group captures only the LAST repetition

- The full match contains everything

Regex engine overwrites the group on every loop.

### 7.2) `(?:pattern)`
**Non-capturing groups**: It groups the pattern without capturing it for extraction.<br>
The difference between capturing and non-capturing groups is that capturing groups store the matched text for later use, while non-capturing groups do not.
No `match.group()` is available for non-capturing groups.

In [241]:
text = "abababab ab abab"
pattern = r'(?:ab)+'
matches = re.findall(pattern, text)
print(f"{pattern} matches: {matches}")

(?:ab)+ matches: ['abababab', 'ab', 'abab']


In [245]:
text = "abcd acbd"
pattern1 = r'(ab)(cd)'    # Two capturing groups
pattern2 = r'(?:ab)(cd)'  # One capturing group

match1 = re.search(pattern1, text)
match2 = re.search(pattern2, text)

print("Pattern (ab)(cd):")
if match1:
    print(f"  Full match: {match1.group(0)}")   
    print(f"  Group 1: {match1.group(1)}")     
    print(f"  Group 2: {match1.group(2)}")      
    print(f"  All groups: {match1.groups()}")  

print("\nPattern (?:ab)(cd):")
if match2:
    print(f"  Full match: {match2.group(0)}")   
    print(f"  Group 1: {match2.group(1)}")     
    print(f"  All groups: {match2.groups()}") 

Pattern (ab)(cd):
  Full match: abcd
  Group 1: ab
  Group 2: cd
  All groups: ('ab', 'cd')

Pattern (?:ab)(cd):
  Full match: abcd
  Group 1: cd
  All groups: ('cd',)


In [267]:
text = "unhappy undo unable until"
pattern = r"\b(?:un)+(happy|do|able)\b"

matches = re.findall(pattern, text)
print(f"{pattern} matches: {matches}") # only the captured parts

print("\nDetailed matches:")
for i in re.finditer(pattern, text):
    print(f"    Full match: '{i.group(0)}', Captured: '{i.group(1)}'") 

\b(?:un)+(happy|do|able)\b matches: ['happy', 'do', 'able']

Detailed matches:
    Full match: 'unhappy', Captured: 'happy'
    Full match: 'undo', Captured: 'do'
    Full match: 'unable', Captured: 'able'


In [151]:
# want sub parts before and after
text = "unhappy undo unable until"
pattern = r"\b(un)+(happy|do|able)\b"

matches = re.findall(pattern, text)
print(f"{pattern} matches: {matches}") # only the captured parts

print("\nDetailed matches:")
for i in re.finditer(pattern, text):
    print(f"    Full match: '{i.group(0)}', Captured: '{i.group(1)}' and '{i.group(2)}'") 

\b(un)+(happy|do|able)\b matches: [('un', 'happy'), ('un', 'do'), ('un', 'able')]

Detailed matches:
    Full match: 'unhappy', Captured: 'un' and 'happy'
    Full match: 'undo', Captured: 'un' and 'do'
    Full match: 'unable', Captured: 'un' and 'able'


In [260]:
# want whole words, no capturing groups
text = "unhappy undo unable until"
pattern = r"\b(?:un)+(?:happy|do|able)\b"

matches = re.findall(pattern, text)
print(f"{pattern} matches: {matches}") # all matched parts 

\b(?:un)+(?:happy|do|able)\b matches: ['unhappy', 'undo', 'unable']


### 7.3) `(?P<name>...)`
**Named capturing groups**: It allows us to assign a name to a capturing group for easier access later.<br>

In [286]:
text3 = "+91-1234567890 and 919876543210"
pattern3 = r'(?P<country>\+*\d{2})\-*(?P<mobile>\d+)'
match3 = re.finditer(pattern3, text3)
for match in match3:
    print(f"Named groups match: {match.groupdict()}")

Named groups match: {'country': '+91', 'mobile': '1234567890'}
Named groups match: {'country': '91', 'mobile': '9876543210'}


### 7.4) `(x|y)`
**Alternation**: It allows us to match one of several patterns using the `|` operator.

In [105]:
text4 = "red banther bread rant bear"
pattern4 = r'(re|ba)'
matches4 = re.findall(pattern4, text4)
print(f"   {pattern4} matches: {matches4}")  

   (re|ba) matches: ['re', 'ba', 're']


### 7.5) Backreference `\n` (numeric)
**Numeric backreference**: It allows us to refer to a previously captured group by its number.

In [141]:
# Find accidentally repeated consecutive words (common typo)
text = "The the quick brown fox jumps over over the lazy dog dog."
pattern = r'\b(\w+)\s+\1\b'  # \1 refers to the first captured group

repeated_words = re.finditer(pattern, text, flags=re.IGNORECASE)
for i in repeated_words:
    print(f"Found repeated word: '{i.group(0)}' at position {i.start()}-{i.end()}")


# Fix the repeated words
fixed_text = re.sub(pattern, r'\1', text, flags=re.IGNORECASE)  # Replace duplicate with single
print(f"Fixed: {fixed_text}")
# Output: "The quick brown fox jumps over the lazy dog."

Found repeated word: 'The the' at position 0-7
Found repeated word: 'over over' at position 30-39
Found repeated word: 'dog dog' at position 49-56
Fixed: The quick brown fox jumps over the lazy dog.


In [143]:
# palindromes detection
text = "radar civic noon test hello racecar"
pattern = r'\b(\w)(\w)\w*\2\1\b'

matches = list(re.finditer(pattern, text))
print("Palindromes found: ")
for match in matches:
    print(match)

Palindromes found: 
<re.Match object; span=(0, 5), match='radar'>
<re.Match object; span=(6, 11), match='civic'>
<re.Match object; span=(12, 16), match='noon'>
<re.Match object; span=(28, 35), match='racecar'>


In [ ]:
# Check for properly closed tags
html = "<h1>Title</h1><p>Text</div>\n<h2>Subtitle</h2>"
pattern = r'<(\w+)>[^<]*</\1>'

matches = re.finditer(pattern, html)
for i in matches:
    print(f"Properly closed tags in '{i.group(0)}' is '{i.group(1)}'")

Properly closed tags in '<h1>Title</h1>' is 'h1'
Properly closed tags in '<h2>Subtitle</h2>' is 'h2'


### 7.6) Named backreference (Python syntax: `(?P=name)`)
**Named backreference**: It allows us to refer to a previously captured named group by its name.

In [150]:
text6 = "51245 55 523 51"
pattern6 = r'(?P<first>5)(\d*)(?P=first)'
matches6 = list(re.finditer(pattern6, text6))
print(matches6) 

[<re.Match object; span=(0, 5), match='51245'>, <re.Match object; span=(6, 8), match='55'>]


## 8) Questions for practice:

**Q6:** Extract area codes from phone numbers in various formats.<br>
*Input*: `Call 123-1234, (555) 123-4567 or (987) 555-7890`<br>
*Output*: `['555', '987']`<br>

**Q7:** Separate email addresses into username and domain parts.<br>
*Input*: `"Emails: john@example.com, jane.doe@company.co.uk"`<br>
*Output*: `[('john', 'example.com'), ('jane.doe', 'company.co.uk')]`<br>

**Q8:** Extract tag names and their text content from simple HTML.<br>
*Input*: `"<h1>Title</h1><p>Paragraph text</p><div>Content</div>"`<br>
*Output*: `{'tag': 'h1', 'body': 'Title'}`<br>
            `{'tag': 'p', 'body': 'Paragraph text'}`<br>
            `{'tag': 'div', 'body': 'Content'}`<br>

**Q9:** Parse Log File Entries.<br>
*Input*: `"2023-12-25 10:30:45 ERROR: File not found\n2023-12-25 11:15:20 INFO: User logged in"`<br>
*Output*: `{'datetime': '2023-12-25 10:30:45', 'level': 'ERROR', 'message': 'File not found'}`<br>
`{'datetime': '2023-12-25 11:15:20', 'level': 'INFO', 'message': 'User logged in'}`<br>

**Q10:** Extract currency amounts with their symbols..<br>
*Input*: `"Prices: $19.99, €25.50, ¥1000, invalid$abc"`<br>
*Output*: `[('$', '19.99'), ('€', '25.50'), ('¥', '1000')]`<br>

**Q11:** Extract protocol, domain, and path from URLs.<br>
*Input*: `"Links: https://example.com/page, http://site.org/about/index.html, https://www.google.com/"`<br>
*Output*: `[('https', 'example.com', '/page'), ('http', 'site.org', '/about/index.html'), ('https', 'www.google.com', '')]`<br>

In [5]:
import re
text = "Call 123-1234, (555) 123-4567 or (987) 555-7890"
pattern = r'\(?(\d{3})\)?[- ]?\d{3}-\d{4}'
match3 = re.finditer(pattern, text)
for match in match3:
    print(f"Named groups match: {match.group(1)}")

Named groups match: 555
Named groups match: 987


In [38]:
text = "Emails: john@example.com, jane.doe@company.co.uk"
pattern = r"([\w+\.*]+)[@]([\w+\.*]+)\b"
match3 = re.finditer(pattern, text)
for match in match3:
    print(f"Named groups match: {match.group(1)} and {match.group(2)}")

Named groups match: john and example.com
Named groups match: jane.doe and company.co.uk


In [37]:
text = "<h1>Title</h1><p>Paragraph text</p><div>Content</div>"
pattern = r'<(?P<tag>\w+)>(?P<body>.*?)</\1>'
match3 = re.finditer(pattern, text)
for m in match3:
    print(m.groupdict())

{'tag': 'h1', 'body': 'Title'}
{'tag': 'p', 'body': 'Paragraph text'}
{'tag': 'div', 'body': 'Content'}


In [36]:
text = "2023-12-25 10:30:45 ERROR: File not found\n2023-12-25 11:15:20 INFO: User logged in"
pattern = r'(?P<datetime>\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}:\d{2})\s(?P<level>\w+)\:\s(?P<message>.*)'
match3 = re.finditer(pattern, text)
for m in match3:
    print(m.groupdict())

{'datetime': '2023-12-25 10:30:45', 'level': 'ERROR', 'message': 'File not found'}
{'datetime': '2023-12-25 11:15:20', 'level': 'INFO', 'message': 'User logged in'}


In [ ]:
import regex as re
text = "Prices: €25.52, ¥100, ₹25.5 invalid$abc"
pattern = r'(\p{Sc})(\d+(\.\d{0,2})*)' # \p{Sc} matches currency symbols
match3 = re.finditer(pattern, text)
for match in match3:
    print(f"Named groups match: {match.group(1)} and {match.group(2)}")

Named groups match: € and 25.52
Named groups match: ¥ and 100
Named groups match: ₹ and 25.5


In [75]:
text = "Links: https://example.com/page, http://site.org/about/index.html, https://www.google.com/"
pattern = r"(https|http):\/\/([\w.\-]+)\/([\w./\-]+)*"
match3 = re.finditer(pattern, text)
for match in match3:
    print(f"Named groups match: {match.group(1)} - {match.group(2)} - {match.group(3)}")

Named groups match: https - example.com - page
Named groups match: http - site.org - about/index.html
Named groups match: https - www.google.com - None


## 9) Lookaround
This is an advanced regex feature that allows you to assert whether a pattern is preceded or followed by another pattern, without including that pattern in the match.

| Syntax | Description | Example pattern | Example matches | Example non-matches |
| :--- | :--- | :--- | :--- | :--- |
| `(?=x)` | looks ahead at the next characters without using them in the match | `an(?=an)`<br>`iss(?=ipp)` | b<span style="background-color: #6de0d1;">an</span>ana<br>Miss<span style="background-color: #6de0d1;">iss</span>ippi | band<br>missed |
| `(?!x)` | looks ahead at next characters to not match on | `ai(?!n)` | f<span style="background-color: #6de0d1;">ai</span>l<br>br<span style="background-color: #6de0d1;">ai</span>l | faint<br>train |
| `(?<=x)` | looks at previous characters for a match without using those in the match | `(?<=tr)a` | tr<span style="background-color: #6de0d1;">a</span>il<br>tr<span style="background-color: #6de0d1;">a</span>nslate | bear<br>streak |
| `(?<!x)` | looks at previous characters to not match on | `(?<!tr)a` | be<span style="background-color: #6de0d1;">a</span>r<br>transl<span style="background-color: #6de0d1;">a</span>te | trail<br>strained |

### 9.1) `(?=x)` - Positive Lookahead
In positive lookahead, the pattern matches if the text to the right of the current position matches the pattern.

In [13]:
text1 = "banana Mississippi missed"
pattern1 = r'an(?=ana)|iss(?=issi)'
matches1 = list(re.finditer(pattern1, text1))
print(f"Pattern: {pattern1}")
print(f"Text: '{text1}'")
for match in matches1:
    print(f"  Match: '{match.group(0)}' at position {match.start()}")

Pattern: an(?=ana)|iss(?=issi)
Text: 'banana Mississippi missed'
  Match: 'an' at position 1
  Match: 'iss' at position 8


In [18]:
text_css = "padding: 10px; margin: 2em; width: 100%; height: 50px"
pattern_css = r'\d+(?=px|em)'
css_values = re.findall(pattern_css, text_css)
print(f"Text: '{text_css}'")
print(f"Pattern: '{pattern_css}'")
print(f"Values: {css_values}")

Text: 'padding: 10px; margin: 2em; width: 100%; height: 50px'
Pattern: '\d+(?=px|em)'
Values: ['10', '2', '50']


### 9.2) `(?!x)` - Negative Lookahead

In negative lookahead, the pattern matches if the text to the right of the current position does not match the pattern.


In [23]:
text2 = "fail faint train fraigile"
pattern2 = r'ai(?!n)'
matches2 = list(re.finditer(pattern2, text2))
print(f"Pattern: '{pattern2}'")
print(f"Text: '{text2}'")
for match in matches2:
    print(f"- Match: '{match.group()}' at position {match.start()}")
    print(f"- What follows: '{text2[match.end():match.end()+2]}'")

Pattern: 'ai(?!n)'
Text: 'fail faint train fraigile'
- Match: 'ai' at position 1
- What follows: 'l '
- Match: 'ai' at position 19
- What follows: 'gi'


### 9.3) `(?<=x)` - Positive Lookbehind

In positive lookbehind, the pattern matches if the text to the left of the current position matches the pattern.

In [29]:
text3 = "trail translate bear streak"
pattern3 = r'(?<=tr)a'
matches3 = list(re.finditer(pattern3, text3))
print(f"Pattern: '{pattern3}'")
print(f"Text: '{text3}'")
for match in matches3:
    print(f"- Match: '{match.group()}' at position {match.start()}")
    print(f"- What follows: '{text3[match.start()+1:match.start()+3]}'")

Pattern: '(?<=tr)a'
Text: 'trail translate bear streak'
- Match: 'a' at position 2
- What follows: 'il'
- Match: 'a' at position 8
- What follows: 'ns'


In [32]:
text3 = "Fruits total 10kg in which apple are 5000gm"
pattern3 = r'(?<=\d+)[a-zA-Z]+'
matches3 = list(re.finditer(pattern3, text3))
print(f"Pattern: '{pattern3}'")
print(f"Text: '{text3}'")
for match in matches3:
    print(match)

Pattern: '(?<=\d+)[a-zA-Z]+'
Text: 'Fruits total 10kg in which apple are 5000gm'
<regex.Match object; span=(15, 17), match='kg'>
<regex.Match object; span=(41, 43), match='gm'>


### 9.4) `(?<!x)` - Negative lookbehind

In negative lookbehind, the pattern matches if the text to the left of the current position does not match the pattern.

In [3]:
# Match @mentions but not email addresses
text_mentions = "Hello @john and @jane! Email: john@example.com"
pattern_mentions = r'(?<!\w)@\w+'
mentions = re.findall(pattern_mentions, text_mentions)
print(f"Text: '{text_mentions}'")
print(f"Pattern: '{pattern_mentions}'")
print(f"Mentions: {mentions}")

Text: 'Hello @john and @jane! Email: john@example.com'
Pattern: '(?<!\w)@\w+'
Mentions: ['@john', '@jane']


## 10) Questions for practice:

**Q12:** Extract dollar amounts (without $):<br>
*Input:* `"Prices: $19.99, $25, ₹100.50, Total: $200 €300"`<br>
*Output:* `["19.99", "25", "200"]`<br>

**Q13:** Find all words ending with 'ing' but NOT 'thing' or 'nothing'<br>
*Input:* `"running jumping something anything singing nothing everything"`<br>
*Output:* `["running", "jumping", "singing"]`<br>

**Q14:** Extract 'cat' when not 'wildcat' or 'catfish'<br>
*Input:* `"cat wildcat catfish bobcat catalog"`<br>
*Output:* `["cat"]`<br>

**Q15:** Write a regex that matches only non-image URLs, with these rules:<br>
*Input:* `"Check these links: https://example.com/page, https://example.com/photo.jpeg, http://site.org/docs/report.pdf, https://cdn.site.net/icon.svg, https://blog.site.ai/post"`<br>
*Output:* `["https://example.com/page", "http://site.org/docs/report.pdf", "https://blog.site.ai/post"]`<br>

In [33]:
text_money = "Prices: $19.99, $25, ₹100.50, Total: $200 €300"
pattern_money = r'(?<=\$)\d+(?:\.\d+)?'
amounts = re.findall(pattern_money, text_money)
print(f"Text: '{text_money}'")
print(f"Pattern: '(?<=\\$)\\d+(?:\\.\\d+)?'")
print(f"Amounts: {amounts}")

Text: 'Prices: $19.99, $25, ₹100.50, Total: $200 €300'
Pattern: '(?<=\$)\d+(?:\.\d+)?'
Amounts: ['19.99', '25', '200']


In [74]:
text = "running jumping something anything singing nothing everything"
pattern = r'\b(?!\w*thing.*)\w+ing\b'
match = re.finditer(pattern, text)
print(f"Text: '{text}'")
print(f"Pattern: '{pattern}'")
for i in match:
    print(i)

Text: 'running jumping something anything singing nothing everything'
Pattern: '\b(?!\w*thing.*)\w+ing\b'
<re.Match object; span=(0, 7), match='running'>
<re.Match object; span=(8, 15), match='jumping'>
<re.Match object; span=(35, 42), match='singing'>


In [ ]:
text_cat = "cat wildcat catfish bobcat catalog"
pattern_cat = r'(?<!\w)(?<!wild)cat(?!fish|alog)\b'
cat_matches = re.findall(pattern_cat, text_cat)
print(f"Text: '{text_cat}'")
print(f"Matches: {cat_matches}")

Text: 'cat wildcat catfish bobcat catalog'
Pattern: uses lookbehind AND lookahead
Matches: ['cat']


In [98]:
import re

text = """Check these links:
https://example.com/page,
https://example.com/photo.jpeg
http://site.org/docs/report.pdf)
https://cdn.site.net/icon.svg,
https://blog.site.ai/post
"""

pattern = r'https?://(?![^\s\,\)\]]*\.(?:jpg|jpeg|png|gif|svg)\b)[^\s\,\)\]]+'

matches = re.finditer(pattern, text)

print("Input text:")
print(text)
print("Matched URLs:")
for m in matches:
    print(m)


Input text:
Check these links:
https://example.com/page,
https://example.com/photo.jpeg
http://site.org/docs/report.pdf)
https://cdn.site.net/icon.svg,
https://blog.site.ai/post

Matched URLs:
<re.Match object; span=(19, 43), match='https://example.com/page'>
<re.Match object; span=(76, 107), match='http://site.org/docs/report.pdf'>
<re.Match object; span=(140, 165), match='https://blog.site.ai/post'>
